# Credit Card Analytics — Interactive Notebook

**How to use:**
1. Edit the **⚙️ Configuration** cell below with your desired settings.
2. Run **Setup** to load data and print the analysis periods.
3. (Optional) Run **Data Preview** to inspect the dataset.
4. Run **Create Agents** and **Run Analysis** to launch the multi-agent pipeline.

## ⚙️ Configuration
**Edit this cell before running anything else.**

In [ ]:
# ── Reference date ────────────────────────────────────────────────────────────
# Analysis covers the month PRIOR to this date.
# e.g. "2015-10-01"  →  analyzes September 2015
REFERENCE_DATE = "2015-10-01"

# ── Data source ───────────────────────────────────────────────────────────────
# Filename inside the data/ folder, OR set CSV_PATH to a full path.
DATA_FILE = "India_cc_transactions.csv"   # e.g. "US_cc_transactions.csv"
CSV_PATH  = None                          # e.g. "/absolute/path/to/my_data.csv"

# ── Dimensions to decompose ───────────────────────────────────────────────────
# Pick any subset of the available dimension columns.
# Available: "Exp Type", "City", "Card Type"
DIMENSIONS = ["Exp Type", "Card Type"]    # remove "City" for a quicker run

# ── Optional: pin the drill-down segment ─────────────────────────────────────
# Leave both as None and the agents will pick the biggest CTG mover.
# e.g. DRILL_DIMENSION = "Exp Type";  DRILL_SEGMENT = "Food"
DRILL_DIMENSION = None
DRILL_SEGMENT   = None

# ── Model ─────────────────────────────────────────────────────────────────────
OPENAI_MODEL = "gpt-4o"

## Setup

In [ ]:
import os, sys

# Add code/ to path so we can import tools, agents, prompts
CODE_DIR = os.path.join(os.getcwd(), "code")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from dotenv import load_dotenv
load_dotenv()

import tools
tools.configure(
    reference_date = REFERENCE_DATE,
    data_file      = DATA_FILE,
    csv_path       = CSV_PATH,   # overrides DATA_FILE if set
)

print(f"Reference date : {tools.REFERENCE_DATE.strftime('%Y-%m-%d')}")
print(f"Analyzing      : {tools._period_label(*tools.CURRENT_MONTH)}")
print(f"Prior month    : {tools._period_label(*tools.PRIOR_MONTH)}")
print(f"Prior year     : {tools._period_label(*tools.PRIOR_YEAR_MONTH)}")
print(f"Rolling window : {tools.ROLLING_START.strftime('%Y-%m-%d')} → {tools.ROLLING_END.strftime('%Y-%m-%d')}")
print(f"Dimensions     : {DIMENSIONS}")
print(f"Data file      : {tools.DEFAULT_CSV}")

## Data Preview

In [ ]:
import json

schema = json.loads(tools.get_schema_info())

print(f"Rows loaded  : {schema['row_count']:,}")
print(f"Date range   : {schema['date_range']['min']} → {schema['date_range']['max']}")
print(f"\nAmount stats :")
for k, v in schema['amount_stats'].items():
    print(f"  {k:12s}: {v:,.2f}" if isinstance(v, float) else f"  {k:12s}: {v:,}")

print(f"\nUnique values per dimension:")
for dim, vals in schema['unique_values'].items():
    print(f"  {dim}: {vals}")

## Quick Tool Test *(optional — run individual tools without agents)*

In [ ]:
# Uncomment any line to inspect raw tool output

# print(tools.get_overall_monthly_summary())
# print(tools.get_dimension_decomposition("Exp Type"))
# print(tools.get_dimension_decomposition("Card Type"))
# print(tools.get_rolling_average())
# print(tools.drill_down_segment("Exp Type", "Food"))

## Run Multi-Agent Analysis

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from agents import create_analyst_agent, create_manager_agent

model_client = OpenAIChatCompletionClient(
    model   = OPENAI_MODEL,
    api_key = os.environ.get("OPENAI_API_KEY"),
)

analyst = create_analyst_agent(model_client, dimensions=DIMENSIONS)
manager = create_manager_agent(model_client, dimensions=DIMENSIONS)

termination = TextMentionTermination("APPROVED") | MaxMessageTermination(10)
team = RoundRobinGroupChat(
    participants=[analyst, manager],
    termination_condition=termination,
)

print("Agents ready.")

In [ ]:
from IPython.display import display, Markdown

current_period = tools._period_label(*tools.CURRENT_MONTH)
dims_label     = ", ".join(DIMENSIONS)

drill_hint = ""
if DRILL_DIMENSION and DRILL_SEGMENT:
    drill_hint = f"\n   For the drill-down, focus on: {DRILL_DIMENSION} = '{DRILL_SEGMENT}'."

task = f"""Analyze credit card transaction data for {current_period}.
Reference date is {tools.REFERENCE_DATE.strftime('%B %d, %Y')}.

Run the full monthly analysis:
1. Confirm schema and analysis periods via get_schema_info
2. Overall spend: MoM delta, % change, transaction volume
3. YoY + CTG decomposition by: {dims_label}
4. 7-day rolling average and rolling avg YoY
5. Identify the biggest CTG mover across the analyzed dimensions and drill into it.{drill_hint}
"""

display(Markdown(f"**Task sent to agents:**\n```\n{task.strip()}\n```"))
print("-" * 60)

async for message in team.run_stream(task=task):
    if hasattr(message, "source") and hasattr(message, "content"):
        display(Markdown(f"---\n### {message.source.upper()}\n\n{message.content}"))